In [ ]:
# wmj-20260119
# MHR→SMPL(X)

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
import numpy as np
from mhr.mhr import MHR
from conversion import Conversion
import smplx

from example import DEMO
demo=DEMO()
device = demo._device
print(device)
smplx_model_file="./data/SMPLX_NEUTRAL.npz"

# Initialize models
mhr_model = MHR.from_files(lod=1, device=device)
smplx_model = smplx.SMPLX(model_path=smplx_model_file, gender="neutral", use_pca=False, flat_hand_mean=True).to(device)


# Create converter
converter = Conversion(
    mhr_model=mhr_model,
    smpl_model=smplx_model,
    method="pytorch"  # or "pymomentum"
)

cpu


/data1/miniconda3/envs/MHR_env/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


## 使用scene npy (可选)

In [7]:
# 加载 mhr_data数据
path= "/data1/wmj/PVCP/sam3dbody_output/scene_npy/S000.npy"
obj = np.load(path, allow_pickle=True)
print(type(obj), obj.dtype, obj.shape)
data = obj.item()
print(type(data))
mhr_data = {
    k: torch.from_numpy(v) for k, v in data.items()
}

<class 'numpy.ndarray'> object ()
<class 'dict'>


## 使用scene json

In [2]:
import json
import numpy as np
import torch
import re
from typing import Dict

def json_to_stacked_tensors(
    json_path: str,
    device: str = "cpu",
    sort_by_frame: bool = True,
) -> Dict[str, torch.Tensor]:
    """
    Convert a json file (frame -> {lbs, iden, expr}) into stacked torch tensors.

    Args:
        json_path: path to json file
        device: 'cpu', 'cuda', 'cuda:0', etc.
        sort_by_frame: whether to sort frames by name / index

    Returns:
        dict with torch.Tensor:
            {
              "lbs_model_params": (N, 204),
              "identity_coeffs":  (N, 45),
              "face_expr_coeffs": (N, 72),
            }
    """

    # -------- load json --------
    with open(json_path, "r", encoding="utf-8") as f:
        frame_dict = json.load(f)

    assert isinstance(frame_dict, dict), "JSON must be a dict: frame -> params"

    # -------- sort frames (optional) --------
    frames = list(frame_dict.keys())

    if sort_by_frame:
        # 优先按 frame 名字里的数字排序，如 frame_000123.png
        def extract_number(name):
            m = re.search(r"(\d+)", name)
            return int(m.group(1)) if m else -1

        frames = sorted(frames, key=extract_number)

    # -------- collect data --------
    lbs_list, iden_list, expr_list = [], [], []

    for frame in frames:
        v = frame_dict[frame]

        # 兼容不同字段命名
        lbs  = v.get("lbs_model_params", v.get("lbs"))
        iden = v.get("identity_coeffs",  v.get("iden"))
        expr = v.get("face_expr_coeffs", v.get("expr"))

        if lbs is None or iden is None or expr is None:
            raise KeyError(f"Missing keys in frame: {frame}")

        lbs  = np.asarray(lbs,  dtype=np.float32).squeeze()
        iden = np.asarray(iden, dtype=np.float32).squeeze()
        expr = np.asarray(expr, dtype=np.float32).squeeze()

        lbs_list.append(lbs)
        iden_list.append(iden)
        expr_list.append(expr)

    # -------- stack --------
    lbs_arr  = np.stack(lbs_list, axis=0)    # (N, 204)
    iden_arr = np.stack(iden_list, axis=0)   # (N, 45)
    expr_arr = np.stack(expr_list, axis=0)   # (N, 72)

    # -------- numpy -> torch --------
    tensor_data = {
        "lbs_model_params": torch.from_numpy(lbs_arr).to(device),
        "identity_coeffs":  torch.from_numpy(iden_arr).to(device),
        "face_expr_coeffs": torch.from_numpy(expr_arr).to(device),
    }

    return tensor_data


mhr_data = json_to_stacked_tensors("/home/guest/wmj/Projects/sam-3d-body/data/PVCP/sam3dbody_output/scene_json/S000.json")

mhr_data

{'lbs_model_params': tensor([[0.0000, -0.0000, -0.0000,  ..., 0.1053, 0.1299, 0.0072],
         [0.0000, -0.0000, -0.0000,  ..., 0.1055, 0.1295, 0.0077],
         [0.0000, -0.0000, -0.0000,  ..., 0.1081, 0.1324, 0.0085],
         ...,
         [0.0000, 0.0000, -0.0000,  ..., 0.0888, 0.1195, 0.0274],
         [0.0000, 0.0000, -0.0000,  ..., 0.0679, 0.1117, 0.0007],
         [0.0000, 0.0000, -0.0000,  ..., 0.0601, 0.1044, 0.0063]]),
 'identity_coeffs': tensor([[-1.7592,  1.1734, -0.4277,  ...,  0.0841,  0.0885,  0.0317],
         [-1.6973,  1.1341, -0.4151,  ...,  0.0808,  0.0869,  0.0347],
         [-1.6050,  1.1136, -0.3680,  ...,  0.0910,  0.0873,  0.0326],
         ...,
         [-2.1532,  1.3041, -0.4080,  ...,  0.0417,  0.0565, -0.0459],
         [-1.9830,  1.3430, -0.3822,  ...,  0.0541,  0.0113,  0.0677],
         [-1.9077,  1.3402, -0.4105,  ...,  0.0921,  0.0702, -0.0402]]),
 'face_expr_coeffs': tensor([[0., -0., 0.,  ..., -0., -0., -0.],
         [0., -0., 0.,  ..., -0., -0., 

In [3]:

# Convert MHR back to SMPLX
smplx_results = converter.convert_mhr2smpl(
    mhr_parameters=mhr_data,
    return_smpl_meshes=True
)

smplx_par0=smplx_results.result_parameters
for k, v in smplx_par0.items():
    shape = v.shape if hasattr(v, "shape") else None
    print(f"{k:20s} | type={type(v).__name__:15s} | shape={shape}")

100%|██████████| 1/1 [00:00<00:00,  6.22it/s]

global_orient        | type=Tensor          | shape=torch.Size([38, 3])
transl               | type=Tensor          | shape=torch.Size([38, 3])
body_pose            | type=Tensor          | shape=torch.Size([38, 63])
betas                | type=Tensor          | shape=torch.Size([38, 10])
left_hand_pose       | type=Tensor          | shape=torch.Size([38, 45])
right_hand_pose      | type=Tensor          | shape=torch.Size([38, 45])
expression           | type=Tensor          | shape=torch.Size([38, 10])


In [10]:
print(type(smplx_results.result_parameters))
print(len(smplx_results.result_parameters))


<class 'dict'>
7
